# Tool-Use GRPO Training on Colab
**Finetuning Agentic Workflows with RL**

Train Qwen2-0.5B-Instruct to make correct tool calls using GRPO.
- No TINKER needed — runs directly on Colab GPU
- Uses TRL's GRPOTrainer
- Binary reward: correct JSON tool call = 1, else = 0

In [ ]:
# Install dependencies
!pip install -q trl>=0.12.0 transformers>=4.45.0 peft>=0.13.0 datasets accelerate bitsandbytes wandb
!pip install -q math_verify latex2sympy2_extended

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import json
import random
from dataclasses import dataclass
from typing import List, Dict

# Available tools
AVAILABLE_TOOLS = [
    {"name": "get_weather", "description": "Get current weather for a location", "parameters": {"location": "string", "unit": "string"}},
    {"name": "calculator", "description": "Perform arithmetic calculations", "parameters": {"expression": "string"}},
    {"name": "set_timer", "description": "Set a countdown timer", "parameters": {"duration_minutes": "number", "label": "string"}},
    {"name": "unit_converter", "description": "Convert between units", "parameters": {"value": "number", "from_unit": "string", "to_unit": "string"}},
    {"name": "translator", "description": "Translate text between languages", "parameters": {"text": "string", "source_lang": "string", "target_lang": "string"}},
    {"name": "web_search", "description": "Search the web", "parameters": {"query": "string", "num_results": "number"}},
    {"name": "send_email", "description": "Send an email", "parameters": {"to": "string", "subject": "string", "body": "string"}},
    {"name": "file_manager", "description": "Manage files", "parameters": {"action": "string", "path": "string"}},
    {"name": "reminder", "description": "Set a reminder", "parameters": {"message": "string", "time": "string"}},
    {"name": "note_taker", "description": "Create and manage notes", "parameters": {"action": "string", "title": "string", "content": "string"}},
    {"name": "currency_converter", "description": "Convert currencies", "parameters": {"amount": "number", "from_currency": "string", "to_currency": "string"}},
    {"name": "recipe_finder", "description": "Search for recipes", "parameters": {"query": "string", "cuisine": "string"}},
    {"name": "password_generator", "description": "Generate secure passwords", "parameters": {"length": "number", "include_symbols": "boolean"}},
    {"name": "stock_price", "description": "Get stock price", "parameters": {"ticker": "string"}},
    {"name": "dictionary_lookup", "description": "Look up word definitions", "parameters": {"word": "string"}},
]

TOOL_NAMES = {t["name"] for t in AVAILABLE_TOOLS}
TOOL_LIST_STR = json.dumps([{"name": t["name"], "description": t["description"], "parameters": t["parameters"]} for t in AVAILABLE_TOOLS], indent=2)

print(f"Defined {len(AVAILABLE_TOOLS)} tools")

In [ ]:
# Training dataset
TOOL_USE_DATASET = [
    {"prompt": "What's the weather like in London?", "tool": "get_weather", "args": {"location": "London", "unit": "celsius"}},
    {"prompt": "Calculate 15 * 23 + 7", "tool": "calculator", "args": {"expression": "15 * 23 + 7"}},
    {"prompt": "Set a timer for 25 minutes for my workout", "tool": "set_timer", "args": {"duration_minutes": 25, "label": "workout"}},
    {"prompt": "Convert 100 kilometers to miles", "tool": "unit_converter", "args": {"value": 100, "from_unit": "kilometers", "to_unit": "miles"}},
    {"prompt": "Translate 'Hello, how are you?' to Spanish", "tool": "translator", "args": {"text": "Hello, how are you?", "source_lang": "English", "target_lang": "Spanish"}},
    {"prompt": "Search for Python machine learning tutorials", "tool": "web_search", "args": {"query": "Python machine learning tutorials", "num_results": 5}},
    {"prompt": "Send an email to team@company.com about the meeting", "tool": "send_email", "args": {"to": "team@company.com", "subject": "Meeting Update", "body": "The meeting has been rescheduled."}},
    {"prompt": "List files in /home/user/projects", "tool": "file_manager", "args": {"action": "list", "path": "/home/user/projects"}},
    {"prompt": "Remind me to call the dentist at 2pm", "tool": "reminder", "args": {"message": "Call the dentist", "time": "14:00"}},
    {"prompt": "Create a note titled 'Ideas' about the new project", "tool": "note_taker", "args": {"action": "create", "title": "Ideas", "content": "New project brainstorm"}},
    {"prompt": "Convert 500 euros to US dollars", "tool": "currency_converter", "args": {"amount": 500, "from_currency": "EUR", "to_currency": "USD"}},
    {"prompt": "Find me a chicken stir fry recipe", "tool": "recipe_finder", "args": {"query": "chicken stir fry", "cuisine": "Asian"}},
    {"prompt": "Generate a 20-character password with symbols", "tool": "password_generator", "args": {"length": 20, "include_symbols": True}},
    {"prompt": "What's the stock price of AAPL?", "tool": "stock_price", "args": {"ticker": "AAPL"}},
    {"prompt": "Define the word 'ubiquitous'", "tool": "dictionary_lookup", "args": {"word": "ubiquitous"}},
    {"prompt": "What's the temperature in Tokyo in fahrenheit?", "tool": "get_weather", "args": {"location": "Tokyo", "unit": "fahrenheit"}},
    {"prompt": "Calculate the square root of 144", "tool": "calculator", "args": {"expression": "sqrt(144)"}},
    {"prompt": "Set a 5 minute timer for tea", "tool": "set_timer", "args": {"duration_minutes": 5, "label": "tea"}},
    {"prompt": "Convert 72 fahrenheit to celsius", "tool": "unit_converter", "args": {"value": 72, "from_unit": "fahrenheit", "to_unit": "celsius"}},
    {"prompt": "Translate 'Thank you' to Japanese", "tool": "translator", "args": {"text": "Thank you", "source_lang": "English", "target_lang": "Japanese"}},
    {"prompt": "Search for latest AI news 2026", "tool": "web_search", "args": {"query": "latest AI news 2026", "num_results": 10}},
    {"prompt": "Email john@example.com about the project deadline", "tool": "send_email", "args": {"to": "john@example.com", "subject": "Project Deadline", "body": "Reminder about the upcoming deadline."}},
    {"prompt": "Read the file at /home/user/notes.txt", "tool": "file_manager", "args": {"action": "read", "path": "/home/user/notes.txt"}},
    {"prompt": "Remind me to submit the report by 5pm", "tool": "reminder", "args": {"message": "Submit the report", "time": "17:00"}},
    {"prompt": "How much is 1000 GBP in INR?", "tool": "currency_converter", "args": {"amount": 1000, "from_currency": "GBP", "to_currency": "INR"}},
    {"prompt": "Find an Italian pasta recipe", "tool": "recipe_finder", "args": {"query": "pasta", "cuisine": "Italian"}},
    {"prompt": "Generate a 12 character password without symbols", "tool": "password_generator", "args": {"length": 12, "include_symbols": False}},
    {"prompt": "What's TSLA stock price?", "tool": "stock_price", "args": {"ticker": "TSLA"}},
    {"prompt": "What does 'ephemeral' mean?", "tool": "dictionary_lookup", "args": {"word": "ephemeral"}},
    {"prompt": "Weather in New York City?", "tool": "get_weather", "args": {"location": "New York City", "unit": "fahrenheit"}},
    {"prompt": "What is 2^10?", "tool": "calculator", "args": {"expression": "2^10"}},
    {"prompt": "Convert 5 pounds to kilograms", "tool": "unit_converter", "args": {"value": 5, "from_unit": "pounds", "to_unit": "kilograms"}},
    {"prompt": "Translate 'Good morning' to French", "tool": "translator", "args": {"text": "Good morning", "source_lang": "English", "target_lang": "French"}},
    {"prompt": "Look up the word 'serendipity'", "tool": "dictionary_lookup", "args": {"word": "serendipity"}},
    {"prompt": "Search for React tutorial for beginners", "tool": "web_search", "args": {"query": "React tutorial beginners", "num_results": 5}},
    {"prompt": "Convert 250 USD to JPY", "tool": "currency_converter", "args": {"amount": 250, "from_currency": "USD", "to_currency": "JPY"}},
    {"prompt": "Set a 10 minute meditation timer", "tool": "set_timer", "args": {"duration_minutes": 10, "label": "meditation"}},
    {"prompt": "Create a note about grocery list", "tool": "note_taker", "args": {"action": "create", "title": "Grocery List", "content": "Milk, eggs, bread, butter"}},
    {"prompt": "Generate a 24 character secure password", "tool": "password_generator", "args": {"length": 24, "include_symbols": True}},
    {"prompt": "Weather forecast for Berlin?", "tool": "get_weather", "args": {"location": "Berlin", "unit": "celsius"}},
]

print(f"Training dataset: {len(TOOL_USE_DATASET)} examples")

In [ ]:
# Build the reward function
SYSTEM_PROMPT = f"""You are a helpful assistant that can use tools. When the user asks you to do something, respond with a JSON tool call.

Available tools:
{TOOL_LIST_STR}

Respond with ONLY a valid JSON object in this format:
{{"tool": "<tool_name>", "arguments": {{<key>: <value>, ...}}}}

Do not include any other text. Just the JSON."""

def tool_use_reward(completions, prompts, expected_tools, **kwargs):
    """GRPO reward function for tool-use. Binary: 1.0 or 0.0."""
    rewards = []
    for completion, expected_tool in zip(completions, expected_tools):
        text = completion[0]["content"] if isinstance(completion[0], dict) else str(completion)
        text = text.strip()
        
        # Strip markdown
        if text.startswith("```"):
            text = text.split("\n", 1)[1] if "\n" in text else text[3:]
            if text.endswith("```"): text = text[:-3]
            text = text.strip()
        
        try:
            parsed = json.loads(text)
            if not isinstance(parsed, dict) or "tool" not in parsed:
                rewards.append(0.0)
                continue
            if parsed["tool"] not in TOOL_NAMES:
                rewards.append(0.0)
                continue
            if parsed["tool"] != expected_tool:
                rewards.append(0.25)  # Partial credit for valid but wrong tool
                continue
            if "arguments" not in parsed or not isinstance(parsed["arguments"], dict):
                rewards.append(0.5)  # Right tool, no args
                continue
            rewards.append(1.0)  # Full credit
        except (json.JSONDecodeError, KeyError, TypeError):
            rewards.append(0.0)
    
    return rewards

print("Reward function defined")

In [ ]:
# Prepare dataset for TRL GRPOTrainer
from datasets import Dataset

def format_prompt(item):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["prompt"]},
        ],
        "expected_tool": item["tool"],
    }

formatted = [format_prompt(item) for item in TOOL_USE_DATASET]
dataset = Dataset.from_list(formatted)

# Split into train/eval
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
# Load model with LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters() / 1e6:.1f}M")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"LoRA config: rank={lora_config.r}, alpha={lora_config.lora_alpha}")

In [ ]:
# Configure GRPO training
# Match the hyperparameters from TINKER experiments
training_args = GRPOConfig(
    output_dir="./tool_use_grpo_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # effective batch = 32
    learning_rate=4e-5,
    warmup_steps=10,
    logging_steps=1,
    save_steps=10,
    eval_steps=10,
    max_completion_length=256,
    num_generations=16,  # group size = 16, same as TINKER
    temperature=0.7,
    bf16=True,
    report_to="wandb",
    run_name="tool-use-grpo-qwen0.5b",
)

print("GRPO config ready")
print(f"  Group size: {training_args.num_generations}")
print(f"  LR: {training_args.learning_rate}")
print(f"  Max completion: {training_args.max_completion_length}")

In [ ]:
# Optional: login to wandb for tracking
import wandb
wandb.login()  # Will prompt for API key or use WANDB_API_KEY env var

In [ ]:
# Create GRPO Trainer
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    reward_funcs=tool_use_reward,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("GRPOTrainer initialized!")

In [ ]:
# Train!
print("Starting GRPO training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save the model
trainer.save_model("./tool_use_grpo_final")
tokenizer.save_pretrained("./tool_use_grpo_final")
print("Model saved to ./tool_use_grpo_final")

In [ ]:
# Evaluate: compare base vs GRPO-trained
from peft import PeftModel

# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)

eval_prompts = [
    ("Convert 5 miles to kilometers", "unit_converter"),
    ("What does 'ephemeral' mean?", "dictionary_lookup"),
    ("What's the current price of Tesla stock?", "stock_price"),
    ("Translate 'Good morning' to French", "translator"),
    ("Remind me to pick up groceries at 5pm", "reminder"),
    ("Set an alarm for 6:30 AM tomorrow", "set_timer"),
    ("Create a note titled 'Meeting Notes'", "note_taker"),
    ("Search the web for AI news", "web_search"),
    ("Send an email to john@example.com", "send_email"),
    ("List files in /home/user/documents", "file_manager"),
    ("Convert 100 USD to Euros", "currency_converter"),
    ("Find me an Italian pasta recipe", "recipe_finder"),
    ("Generate a 16-character password", "password_generator"),
    ("What's the weather in Tokyo?", "get_weather"),
    ("Calculate 15 * 23 + 7", "calculator"),
]

def evaluate_model(model, tokenizer, prompts):
    correct = 0
    results = []
    for prompt_text, expected_tool in prompts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                     pad_token_id=tokenizer.pad_token_id)
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        
        try:
            parsed = json.loads(response.strip())
            tool_correct = parsed.get("tool") == expected_tool
        except:
            tool_correct = False
        
        correct += int(tool_correct)
        results.append({"prompt": prompt_text, "expected": expected_tool, 
                       "response": response[:100], "correct": tool_correct})
    
    return correct / len(prompts), results

print("Evaluating base model...")
base_acc, base_results = evaluate_model(base_model, tokenizer, eval_prompts)
print(f"Base model accuracy: {base_acc:.1%}")

print("\nEvaluating GRPO model...")
grpo_acc, grpo_results = evaluate_model(trainer.model, tokenizer, eval_prompts)
print(f"GRPO model accuracy: {grpo_acc:.1%}")

print(f"\nImprovement: {grpo_acc - base_acc:+.1%}")

In [ ]:
# Push to HuggingFace Hub
from huggingface_hub import login
login()  # Will prompt for token

trainer.model.push_to_hub("arvindcr4/tool-call-grpo-qwen0.5b")
tokenizer.push_to_hub("arvindcr4/tool-call-grpo-qwen0.5b")
print("Pushed to HuggingFace Hub!")

In [ ]:
# Print detailed comparison
print("="*60)
print("DETAILED COMPARISON: Base vs GRPO")
print("="*60)
for b, g in zip(base_results, grpo_results):
    status_b = "PASS" if b["correct"] else "FAIL"
    status_g = "PASS" if g["correct"] else "FAIL"
    marker = ">>>" if status_g != status_b else "   "
    print(f"{marker} {b['prompt'][:40]:40s} Base:{status_b} GRPO:{status_g}")
print(f"\nBase: {base_acc:.1%} | GRPO: {grpo_acc:.1%} | Delta: {grpo_acc-base_acc:+.1%}")